In [ ]:
# ── CELL 1: MOUNT & INSTALL ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
# !pip install statsmodels -q
#!pip install regex
#import regex as re

Mounted at /content/drive


In [ ]:
!pip install scikit-learn

In [ ]:
MASTER_CSV    = '/content/drive/MyDrive/HEC Thesis/Data/Master_Macro_old.csv'
OUTPUT_CSV    = '/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv'
FRED_API_KEY  = 'c5b5ec287d025fecd23b73051ee03c84'
JK_SHOCKS_CSV = '/content/drive/MyDrive/HEC Thesis/Data/fomc_surprises_jk.csv.xlsx'

In [ ]:
"""
append_sentiment_channels.py
════════════════════════════════════════════════════════════════════════════════
Appends six new regressor groups to Master_Macro.csv, each designed to test
a specific information channel that a dictionary-based FOMC sentiment indicator
might be capturing.

NEW COLUMNS ADDED
─────────────────────────────────────────────────────────────────────────────
1. FORWARD GUIDANCE SHIFT
   ffr_path_change      — Δimplied_ffr between consecutive meeting dates

2. INFLATION EXPECTATIONS
   michigan_1yr_inf_exp — Michigan Survey 1yr ahead inflation expectations (FRED: MICH)
   inf_exp_5y5y         — 5yr/5yr forward breakeven inflation (FRED: T5YIFR)
   inf_exp_gap          — michigan_1yr_inf_exp − 2.0

3. FINANCIAL CONDITIONS
   nfci                 — Chicago Fed National Financial Conditions Index (FRED: NFCI)
   sloos_ci_tightening  — SLOOS net % banks tightening C&I lending standards (FRED: DRTSCILM)

4. MP SURPRISE / INFORMATION EFFECT  (Jarociński-Karadi decomposition)
   jk_mp1               — Raw MP1 surprise from fomc_surprises_jk.csv
   jk_mp_shock          — Part of MP1 orthogonal to SP500 (pure policy shock)
   jk_info_shock        — Part of MP1 correlated with SP500 (information effect)

5. STATEMENT NOVELTY
   stmt_novelty_jaccard — 1 − Jaccard similarity vs prior statement
   stmt_novelty_cosine  — 1 − TF-IDF cosine similarity vs prior statement

6. FOMC INTERNAL DISAGREEMENT
   fomc_dissent_count   — Number of dissenting votes at each meeting

SETUP
─────────────────────────────────────────────────────────────────────────────
  1. Set FRED_API_KEY  (free at https://fred.stlouisfed.org/docs/api/)
  2. Set JK_SHOCKS_CSV to the path of fomc_surprises_jk.csv
  3. Set MASTER_CSV and OUTPUT_CSV paths
  4. !pip install scikit-learn  (if not already installed)
"""

import os
import re
import time
import warnings
import requests
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← edit these three lines before running
# ════════════════════════════════════════════════════════════════════════════

MASTER_CSV    = '/content/drive/MyDrive/HEC Thesis/Data/Master_Macro_old.csv'
OUTPUT_CSV    = '/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv'
FRED_API_KEY  = 'c5b5ec287d025fecd23b73051ee03c84'
JK_SHOCKS_CSV = '/content/drive/MyDrive/HEC Thesis/Data/fomc_surprises_jk.csv.xlsx'

# ════════════════════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════════════════════

def fred_series(series_id, api_key, start='2010-01-01'):
    """Fetch a FRED series and return a date-indexed Series."""
    url = (f'https://api.stlouisfed.org/fred/series/observations'
           f'?series_id={series_id}&api_key={api_key}'
           f'&observation_start={start}&file_type=json')
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()['observations']
    s = pd.Series(
        {pd.Timestamp(d['date']): float(d['value'])
         for d in data if d['value'] != '.'},
        name=series_id
    )
    return s.sort_index()


def merge_asof_backward(df_master, series, col_name):
    """
    For each row in df_master, attach the most recent value of `series`
    that is on or before that row's date (no look-ahead).
    """
    s_df = series.reset_index()
    s_df.columns = ['date', col_name]
    s_df['date'] = pd.to_datetime(s_df['date'])

    tmp = pd.DataFrame({'date': pd.to_datetime(df_master['date'])})
    tmp = tmp.sort_values('date').reset_index()   # preserves original index

    merged = pd.merge_asof(tmp, s_df.sort_values('date'),
                           on='date', direction='backward')
    merged = merged.set_index('index').sort_index()
    return merged[col_name]


def print_section(title):
    print(f'\n{"═"*70}')
    print(f'  {title}')
    print(f'{"═"*70}')


# ════════════════════════════════════════════════════════════════════════════
# LOAD MASTER
# ════════════════════════════════════════════════════════════════════════════

print_section('Loading Master_Macro.csv')
df = pd.read_csv(MASTER_CSV, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f'  Loaded: {len(df)} rows × {len(df.columns)} columns')
print(f'  Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Event types: {df["event_type"].value_counts().to_dict()}')

is_meeting = df['event_type'] == 'meeting_date'

# ════════════════════════════════════════════════════════════════════════════
# 1. FORWARD GUIDANCE SHIFT
#    Computed directly from existing implied_ffr — no external data needed.
#    ffr_path_change[t] = implied_ffr at meeting t − implied_ffr at meeting t-1
#    Forward-filled to minutes_date and blackout_date rows.
# ════════════════════════════════════════════════════════════════════════════

print_section('1. Forward Guidance Shift')

meeting_implied = df.loc[is_meeting, ['date', 'implied_ffr']].copy()
meeting_implied = meeting_implied.sort_values('date')
meeting_implied['ffr_path_change'] = meeting_implied['implied_ffr'].diff()

df = df.merge(meeting_implied[['date', 'ffr_path_change']], on='date', how='left')
df['ffr_path_change'] = df['ffr_path_change'].ffill()

print(f'  ✓  ffr_path_change: {df["ffr_path_change"].notna().sum()} non-NaN rows')

# ════════════════════════════════════════════════════════════════════════════
# 2. INFLATION EXPECTATIONS
#    Michigan Survey 1yr (MICH, monthly FRED)
#    5yr/5yr forward breakeven (T5YIFR, daily FRED)
# ════════════════════════════════════════════════════════════════════════════

print_section('2. Inflation Expectations')

# ── Michigan 1yr inflation expectations ──────────────────────────────────────
try:
    mich = fred_series('MICH', FRED_API_KEY)
    df['michigan_1yr_inf_exp'] = merge_asof_backward(df, mich, 'michigan_1yr_inf_exp')
    df['inf_exp_gap']          = df['michigan_1yr_inf_exp'] - 2.0
    print(f'  ✓  michigan_1yr_inf_exp: {df["michigan_1yr_inf_exp"].notna().sum()} non-NaN')
    print(f'  ✓  inf_exp_gap:          {df["inf_exp_gap"].notna().sum()} non-NaN')
except Exception as e:
    print(f'  ✗  MICH fetch failed: {e}')
    df['michigan_1yr_inf_exp'] = np.nan
    df['inf_exp_gap']          = np.nan

# ── 5yr/5yr forward breakeven ─────────────────────────────────────────────────
try:
    t5yifr = fred_series('T5YIFR', FRED_API_KEY)
    df['inf_exp_5y5y'] = merge_asof_backward(df, t5yifr, 'inf_exp_5y5y')
    print(f'  ✓  inf_exp_5y5y: {df["inf_exp_5y5y"].notna().sum()} non-NaN')
except Exception as e:
    print(f'  ✗  T5YIFR fetch failed: {e}')
    df['inf_exp_5y5y'] = np.nan

# ════════════════════════════════════════════════════════════════════════════
# 3. FINANCIAL CONDITIONS
#    NFCI — Chicago Fed weekly financial conditions index (FRED: NFCI)
#    SLOOS — net % banks tightening C&I lending standards (FRED: DRTSCILM)
# ════════════════════════════════════════════════════════════════════════════

print_section('3. Financial Conditions')

# ── NFCI (weekly) ─────────────────────────────────────────────────────────────
try:
    nfci = fred_series('NFCI', FRED_API_KEY)
    df['nfci'] = merge_asof_backward(df, nfci, 'nfci')
    print(f'  ✓  nfci: {df["nfci"].notna().sum()} non-NaN')
except Exception as e:
    print(f'  ✗  NFCI fetch failed: {e}')
    df['nfci'] = np.nan

# ── SLOOS C&I tightening (quarterly) ─────────────────────────────────────────
try:
    sloos = fred_series('DRTSCILM', FRED_API_KEY)
    df['sloos_ci_tightening'] = merge_asof_backward(df, sloos, 'sloos_ci_tightening')
    print(f'  ✓  sloos_ci_tightening: {df["sloos_ci_tightening"].notna().sum()} non-NaN')
except Exception as e:
    print(f'  ✗  DRTSCILM fetch failed: {e}')
    df['sloos_ci_tightening'] = np.nan

# ════════════════════════════════════════════════════════════════════════════
# 4. MP SURPRISE / INFORMATION EFFECT  (Jarociński-Karadi decomposition)
#
#    Uses fomc_surprises_jk.csv which contains:
#      start   — meeting datetime
#      MP1     — change in 1-month-ahead fed funds futures (Kuttner surprise)
#      SP500   — simultaneous S&P 500 surprise
#
#    JK decomposition logic:
#      If rates surprise up AND stocks go down → pure monetary tightening
#      If rates surprise up AND stocks go up   → Fed revealed good news (info effect)
#
#    We implement this via OLS projection:
#      jk_mp_shock   = residual of MP1 ~ SP500  (pure policy, orthogonal to info)
#      jk_info_shock = fitted value of MP1 ~ SP500  (information component)
#
#    Both shocks are meeting-date only (NaN at minutes/blackout dates).
# ════════════════════════════════════════════════════════════════════════════

print_section('4. MP Surprise / Information Effect (Jarociński-Karadi)')

if os.path.exists(JK_SHOCKS_CSV):
    try:
        jk = pd.read_excel(JK_SHOCKS_CSV)

        # ── Parse date ────────────────────────────────────────────────────────
        # The 'start' column contains datetime strings like '1988-02-04 11:30:00'
        # We want only the date part to match with meeting_date rows
        date_col = [c for c in jk.columns if c.lower() in ('start', 'date')][0]
        jk['date'] = pd.to_datetime(jk[date_col]).dt.normalize()  # strip time

        # ── Keep only scheduled FOMC meetings post-2010 ───────────────────────
        if 'description' in jk.columns:
            jk = jk[jk['description'].str.contains('Scheduled', case=False, na=False)]
        jk = jk[jk['date'] >= '2010-01-01'].copy()

        # ── Store raw MP1 ─────────────────────────────────────────────────────
        jk = jk.rename(columns={'MP1': 'jk_mp1'})
        jk['jk_mp1'] = pd.to_numeric(jk['jk_mp1'], errors='coerce')

        # ── JK decomposition via OLS ──────────────────────────────────────────
        sp500_col = [c for c in jk.columns if c.upper() in ('SP500', 'SP500FUT')][0] \
                    if any(c.upper() in ('SP500', 'SP500FUT') for c in jk.columns) \
                    else None

        if sp500_col:
            jk[sp500_col] = pd.to_numeric(jk[sp500_col], errors='coerce')
            clean = jk[['jk_mp1', sp500_col]].dropna()

            ols_jk = sm.OLS(
                clean['jk_mp1'],
                sm.add_constant(clean[sp500_col])
            ).fit()

            jk.loc[clean.index, 'jk_mp_shock']   = ols_jk.resid.values
            jk.loc[clean.index, 'jk_info_shock']  = ols_jk.fittedvalues.values

            print(f'  ✓  JK decomposition: {len(clean)} meetings used')
            print(f'     R² of MP1 ~ SP500: {ols_jk.rsquared:.4f}')
            print(f'     (higher R² = more info effect relative to pure policy)')
        else:
            print(f'  ⚠️  SP500 column not found — storing raw MP1 only')
            print(f'       Available columns: {jk.columns.tolist()}')
            jk['jk_mp_shock']  = jk['jk_mp1']
            jk['jk_info_shock'] = np.nan

        # ── Merge to master (meeting_date rows only) ──────────────────────────
        merge_cols = ['date', 'jk_mp1', 'jk_mp_shock', 'jk_info_shock']
        merge_cols = [c for c in merge_cols if c in jk.columns]

        df = df.merge(jk[merge_cols], on='date', how='left')

        print(f'  ✓  jk_mp1:        {df["jk_mp1"].notna().sum()} non-NaN rows')
        print(f'  ✓  jk_mp_shock:   {df["jk_mp_shock"].notna().sum()} non-NaN rows')
        print(f'  ✓  jk_info_shock: {df["jk_info_shock"].notna().sum()} non-NaN rows')

    except Exception as e:
        print(f'  ✗  JK shocks failed: {e}')
        df['jk_mp1']        = np.nan
        df['jk_mp_shock']   = np.nan
        df['jk_info_shock'] = np.nan
else:
    print(f'  ⚠️  JK shocks file not found: {JK_SHOCKS_CSV}')
    df['jk_mp1']        = np.nan
    df['jk_mp_shock']   = np.nan
    df['jk_info_shock'] = np.nan

# ════════════════════════════════════════════════════════════════════════════
# 5. STATEMENT NOVELTY
#    Scrapes FOMC statements from federalreserve.gov and computes text
#    similarity between consecutive statements. Novelty = 1 − similarity.
#    Takes ~2-3 minutes due to web scraping.
# ════════════════════════════════════════════════════════════════════════════

print_section('5. Statement Novelty (scraping federalreserve.gov)')


def fetch_fomc_statement_urls():
    """Scrape FOMC statement URLs from the Fed historical pages (2011-present)."""
    base = 'https://www.federalreserve.gov'
    url_list = []
    for year in range(2011, 2026):
        page_url = f'{base}/monetarypolicy/fomchistorical{year}.htm'
        try:
            r = requests.get(page_url, timeout=20)
            if r.status_code != 200:
                continue
            matches = re.findall(
                r'href="(/newsevents/pressreleases/monetary(\d{8})a\.htm)"',
                r.text
            )
            for path, date_str in matches:
                try:
                    dt = pd.Timestamp(date_str)
                    url_list.append((dt, base + path))
                except Exception:
                    pass
            time.sleep(0.4)
        except Exception as e:
            print(f'    ⚠️  Year {year}: {e}')
    url_list = sorted(set(url_list), key=lambda x: x[0])
    print(f'  Found {len(url_list)} statement URLs')
    return url_list


def fetch_statement_text(url):
    """Fetch the body text of one FOMC press release."""
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        text = re.sub(r'<[^>]+>', ' ', r.text)
        text = re.sub(r'\s+', ' ', text).strip()
        body = re.search(
            r'For (?:immediate )?release(.*?)(?:For media inquiries|'
            r'Board of Governors|Last Update|Implementation Note)',
            text, re.DOTALL | re.IGNORECASE
        )
        return body.group(1).strip() if body else text[:3000]
    except Exception:
        return None


def jaccard_similarity(text1, text2):
    set1 = set(re.findall(r'\b[a-z]+\b', text1.lower()))
    set2 = set(re.findall(r'\b[a-z]+\b', text2.lower()))
    if not set1 or not set2:
        return np.nan
    return len(set1 & set2) / len(set1 | set2)


try:
    stmt_urls = fetch_fomc_statement_urls()
    if len(stmt_urls) < 5:
        raise ValueError('Too few URLs — check scraper')

    print('  Fetching statement texts (~2 min)...')
    stmt_texts = {}
    for i, (dt, url) in enumerate(stmt_urls):
        text = fetch_statement_text(url)
        if text and len(text) > 100:
            stmt_texts[dt] = text
        if (i + 1) % 10 == 0:
            print(f'    {i+1}/{len(stmt_urls)} fetched...')
        time.sleep(0.3)

    print(f'  Successfully fetched {len(stmt_texts)} texts')

    stmt_dates = sorted(stmt_texts.keys())
    novelty_jaccard = {}
    novelty_cosine  = {}

    # TF-IDF matrix across all statements
    all_texts = [stmt_texts[d] for d in stmt_dates]
    tfidf     = TfidfVectorizer(stop_words='english', max_features=500)
    tfidf_mat = tfidf.fit_transform(all_texts)

    for i in range(1, len(stmt_dates)):
        dt_curr = stmt_dates[i]
        jac = jaccard_similarity(stmt_texts[dt_curr], stmt_texts[stmt_dates[i-1]])
        novelty_jaccard[dt_curr] = 1 - jac if pd.notna(jac) else np.nan
        cos = sk_cosine(tfidf_mat[i], tfidf_mat[i-1])[0, 0]
        novelty_cosine[dt_curr] = 1 - cos

    nov_df = pd.DataFrame({
        'date':                 pd.Series(list(novelty_jaccard.keys())),
        'stmt_novelty_jaccard': pd.Series(list(novelty_jaccard.values())),
        'stmt_novelty_cosine':  pd.Series(list(novelty_cosine.values())),
    })
    nov_df['date'] = pd.to_datetime(nov_df['date'])

    # Match to meeting_date rows (tolerance = 3 days for any date-shift edge cases)
    df_meeting = df[is_meeting][['date']].copy().sort_values('date')
    nov_merged = pd.merge_asof(
        df_meeting,
        nov_df.sort_values('date'),
        on='date', direction='nearest', tolerance=pd.Timedelta('3d')
    )
    df = df.merge(nov_merged, on='date', how='left')

    # Forward-fill to minutes/blackout rows
    df['stmt_novelty_jaccard'] = df['stmt_novelty_jaccard'].ffill()
    df['stmt_novelty_cosine']  = df['stmt_novelty_cosine'].ffill()

    print(f'  ✓  stmt_novelty_jaccard: {df["stmt_novelty_jaccard"].notna().sum()} non-NaN')
    print(f'  ✓  stmt_novelty_cosine:  {df["stmt_novelty_cosine"].notna().sum()} non-NaN')

except Exception as e:
    print(f'  ✗  Statement novelty failed: {e}')
    df['stmt_novelty_jaccard'] = np.nan
    df['stmt_novelty_cosine']  = np.nan

# ════════════════════════════════════════════════════════════════════════════
# 6. FOMC INTERNAL DISAGREEMENT  (dissent count)
#    Parses the same press releases already fetched above.
#    Counts how many members voted against the majority action.
# ════════════════════════════════════════════════════════════════════════════

print_section('6. FOMC Internal Disagreement (Dissent Count)')


TW_FILE = '/content/drive/MyDrive/HEC Thesis/Data/FOMC_Dissents_Data.xlsx'

tw = pd.read_excel(TW_FILE, skiprows=3)
tw.columns = tw.columns.str.strip()

# Already Timestamps — just rename and filter
tw = tw.rename(columns={'FOMC Meeting': 'date'})
tw['date'] = pd.to_datetime(tw['date']).dt.normalize()
tw = tw[tw['date'] >= '2011-01-01'].copy()

tw = tw.rename(columns={
    'Votes Against Action':         'fomc_dissent_count',
    'Number Governors Dissenting':  'fomc_gov_dissent',
    'Number Presidents Dissenting': 'fomc_pres_dissent',
})

print(f"Post-2011 meetings: {len(tw)}")
print(tw[['date', 'fomc_dissent_count']].head(10).to_string())

# Drop old columns and merge
df = df.drop(columns=[c for c in ['fomc_dissent_count', 'fomc_any_dissent',
                                   'fomc_gov_dissent', 'fomc_pres_dissent']
                       if c in df.columns])
df = df.merge(
    tw[['date', 'fomc_dissent_count', 'fomc_gov_dissent', 'fomc_pres_dissent']],
    on='date', how='left'
)
df['fomc_any_dissent'] = (df['fomc_dissent_count'] > 0).astype(float)
df.loc[df['fomc_dissent_count'].isna(), 'fomc_any_dissent'] = np.nan

print(f'\n  ✓  fomc_dissent_count: {df["fomc_dissent_count"].notna().sum()} non-NaN')
dist = df.loc[df['event_type']=='meeting_date','fomc_dissent_count'].value_counts().sort_index().to_dict()
print(f'     Distribution: {dist}')

# ════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print_section('Summary of All New Columns')

new_cols = [
    'ffr_path_change',
    'michigan_1yr_inf_exp', 'inf_exp_5y5y', 'inf_exp_gap',
    'nfci', 'sloos_ci_tightening',
    'jk_mp1', 'jk_mp_shock', 'jk_info_shock',
    'stmt_novelty_jaccard', 'stmt_novelty_cosine',
    'fomc_dissent_count',
]

print(f'\n  {"Column":<30} {"Non-NaN":>8} {"Min":>10} {"Max":>10}')
print(f'  {"─"*62}')
for col in new_cols:
    if col in df.columns:
        s = df[col].dropna()
        if len(s) > 0:
            print(f'  {col:<30} {len(s):>8} {s.min():>10.3f} {s.max():>10.3f}')
        else:
            print(f'  {col:<30} {"0 — check file path / API key":>40}')
    else:
        print(f'  {col:<30} {"NOT IN DATAFRAME":>40}')

# ════════════════════════════════════════════════════════════════════════════
# SAVE
# ════════════════════════════════════════════════════════════════════════════

print_section('Saving')
df.to_csv(OUTPUT_CSV, index=False)
print(f'  ✓  Saved: {OUTPUT_CSV}')
print(f'     {df.shape[0]} rows × {df.shape[1]} columns')


══════════════════════════════════════════════════════════════════════
  Loading Master_Macro.csv
══════════════════════════════════════════════════════════════════════
  Loaded: 363 rows × 222 columns
  Date range: 2011-01-08 → 2025-12-31
  Event types: {'blackout_date': 121, 'meeting_date': 121, 'minutes_date': 121}

══════════════════════════════════════════════════════════════════════
  1. Forward Guidance Shift
══════════════════════════════════════════════════════════════════════
  ✓  ffr_path_change: 359 non-NaN rows

══════════════════════════════════════════════════════════════════════
  2. Inflation Expectations
══════════════════════════════════════════════════════════════════════
  ✓  michigan_1yr_inf_exp: 363 non-NaN
  ✓  inf_exp_gap:          363 non-NaN
  ✓  inf_exp_5y5y: 363 non-NaN

══════════════════════════════════════════════════════════════════════
  3. Financial Conditions
══════════════════════════════════════════════════════════════════════
  ✓  nfci: 363 non-N

In [ ]:
TW_FILE = '/content/drive/MyDrive/HEC Thesis/Data/FOMC_Dissents_Data.xlsx'

tw = pd.read_excel(TW_FILE, skiprows=3)  # skip the header rows
tw.columns = tw.columns.str.strip()

# Parse date from Year + FOMC Meeting columns
tw['date'] = pd.to_datetime(
    tw['FOMC Meeting'].astype(str) + '/' + tw['Year'].astype(str),
    format='%m/%d/%y/%Y', errors='coerce'
)
# Simpler: just parse the meeting column directly
tw['date'] = pd.to_datetime(
    tw['Year'].astype(str) + '-' + tw['FOMC Meeting'].astype(str),
    errors='coerce'
)

# Keep post-2010 and rename columns
tw = tw[tw['date'] >= '2011-01-01'].copy()
tw = tw.rename(columns={
    'Votes Against Action':       'fomc_dissent_count',
    'Number Governors Dissenting':'fomc_gov_dissent',
    'Number Presidents Dissenting':'fomc_pres_dissent',
})

# Drop old columns and merge
df = df.drop(columns=[c for c in ['fomc_dissent_count', 'fomc_any_dissent',
                                    'fomc_gov_dissent', 'fomc_pres_dissent']
                       if c in df.columns])
df = df.merge(
    tw[['date', 'fomc_dissent_count', 'fomc_gov_dissent', 'fomc_pres_dissent']],
    on='date', how='left'
)
df['fomc_any_dissent'] = (df['fomc_dissent_count'] > 0).astype(float)
df.loc[df['fomc_dissent_count'].isna(), 'fomc_any_dissent'] = np.nan

print(f'  ✓  fomc_dissent_count: {df["fomc_dissent_count"].notna().sum()} non-NaN')
print(f'  ✓  fomc_gov_dissent:   {df["fomc_gov_dissent"].notna().sum()} non-NaN')
print(f'  ✓  fomc_pres_dissent:  {df["fomc_pres_dissent"].notna().sum()} non-NaN')
dist = df.loc[df['event_type']=='meeting_date', 'fomc_dissent_count'].value_counts().sort_index().to_dict()
print(f'     Distribution: {dist}')

  ✓  fomc_dissent_count: 0 non-NaN
  ✓  fomc_gov_dissent:   0 non-NaN
  ✓  fomc_pres_dissent:  0 non-NaN
     Distribution: {}


In [ ]:
TW_FILE = '/content/drive/MyDrive/HEC Thesis/Data/FOMC_Dissents_Data.xlsx'

tw = pd.read_excel(TW_FILE, skiprows=3)
tw.columns = tw.columns.str.strip()

# Already Timestamps — just rename and filter
tw = tw.rename(columns={'FOMC Meeting': 'date'})
tw['date'] = pd.to_datetime(tw['date']).dt.normalize()
tw = tw[tw['date'] >= '2011-01-01'].copy()

tw = tw.rename(columns={
    'Votes Against Action':         'fomc_dissent_count',
    'Number Governors Dissenting':  'fomc_gov_dissent',
    'Number Presidents Dissenting': 'fomc_pres_dissent',
})

print(f"Post-2011 meetings: {len(tw)}")
print(tw[['date', 'fomc_dissent_count']].head(10).to_string())

# Drop old columns and merge
df = df.drop(columns=[c for c in ['fomc_dissent_count', 'fomc_any_dissent',
                                   'fomc_gov_dissent', 'fomc_pres_dissent']
                       if c in df.columns])
df = df.merge(
    tw[['date', 'fomc_dissent_count', 'fomc_gov_dissent', 'fomc_pres_dissent']],
    on='date', how='left'
)
df['fomc_any_dissent'] = (df['fomc_dissent_count'] > 0).astype(float)
df.loc[df['fomc_dissent_count'].isna(), 'fomc_any_dissent'] = np.nan

print(f'\n  ✓  fomc_dissent_count: {df["fomc_dissent_count"].notna().sum()} non-NaN')
dist = df.loc[df['event_type']=='meeting_date','fomc_dissent_count'].value_counts().sort_index().to_dict()
print(f'     Distribution: {dist}')

Post-2011 meetings: 124
          date  fomc_dissent_count
733 2011-01-26                 0.0
734 2011-03-15                 0.0
735 2011-04-27                 0.0
736 2011-06-22                 0.0
737 2011-08-09                 3.0
738 2011-09-21                 3.0
739 2011-11-02                 1.0
740 2011-12-13                 1.0
741 2012-01-25                 1.0
742 2012-03-13                 1.0

  ✓  fomc_dissent_count: 121 non-NaN
     Distribution: {0.0: 71, 1.0: 35, 2.0: 9, 3.0: 6}


In [ ]:
TW_FILE = '/content/drive/MyDrive/HEC Thesis/Data/FOMC_Dissents_Data.xlsx'

tw = pd.read_excel(TW_FILE, skiprows=3)
tw.columns = tw.columns.str.strip()

# See exactly what the date column contains
print(tw['FOMC Meeting'].head(10).tolist())

[Timestamp('1936-03-19 00:00:00'), Timestamp('1936-05-25 00:00:00'), Timestamp('1936-11-20 00:00:00'), Timestamp('1937-01-26 00:00:00'), Timestamp('1937-03-15 00:00:00'), Timestamp('1937-04-04 00:00:00'), Timestamp('1937-05-05 00:00:00'), Timestamp('1937-06-09 00:00:00'), Timestamp('1937-11-12 00:00:00'), Timestamp('1937-12-01 00:00:00')]


In [ ]:
TW_FILE = '/content/drive/MyDrive/HEC Thesis/Data/FOMC_Dissents_Data.xlsx'
tw = pd.read_excel(TW_FILE, skiprows=3)
tw.columns = tw.columns.str.strip()
tw = tw.rename(columns={'FOMC Meeting': 'date'})
tw['date'] = pd.to_datetime(tw['date']).dt.normalize()
tw = tw[tw['date'] >= '2011-01-01'].copy()

# Meeting dates from your master
master_dates = set(
    pd.to_datetime(df.reset_index()['date'])
    .dt.normalize()
)
tw_dates = set(tw['date'])

print(f"TW meetings post-2011:     {len(tw_dates)}")
print(f"Master meeting dates:      {len(master_dates)}")
print(f"Matched:                   {len(master_dates & tw_dates)}")
print(f"\nIn TW but NOT in Master:")
for d in sorted(tw_dates - master_dates):
    print(f"  {d.date()}")
print(f"\nIn Master but NOT in TW:")
for d in sorted(master_dates - tw_dates):
    print(f"  {d.date()}")

TW meetings post-2011:     124
Master meeting dates:      360
Matched:                   121

In TW but NOT in Master:
  2020-03-23
  2026-01-28
  2026-03-18

In Master but NOT in TW:
  2011-01-08
  2011-02-16
  2011-03-04
  2011-04-05
  2011-04-14
  2011-05-18
  2011-06-14
  2011-06-29
  2011-07-13
  2011-08-30
  2011-09-15
  2011-10-12
  2011-10-22
  2011-11-23
  2011-11-29
  2012-01-03
  2012-01-16
  2012-02-15
  2012-03-01
  2012-04-03
  2012-04-13
  2012-05-16
  2012-06-12
  2012-07-11
  2012-07-24
  2012-08-22
  2012-08-31
  2012-10-04
  2012-10-14
  2012-11-14
  2012-12-04
  2013-01-02
  2013-01-04
  2013-02-20
  2013-03-08
  2013-04-10
  2013-04-19
  2013-05-22
  2013-06-06
  2013-07-10
  2013-07-17
  2013-08-21
  2013-10-09
  2013-10-18
  2013-11-20
  2013-12-16
  2014-01-03
  2014-01-08
  2014-02-19
  2014-03-05
  2014-04-09
  2014-04-15
  2014-05-21
  2014-06-09
  2014-07-09
  2014-07-10
  2014-08-20
  2014-09-04
  2014-10-08
  2014-10-20
  2014-11-19
  2014-12-03
  2015-01-

In [ ]:
MASTER_CSV = '/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv'
df = pd.read_csv(MASTER_CSV, parse_dates=['date'])
is_meeting = df['event_type'] == 'meeting_date'

check_cols = ['stmt_novelty_jaccard', 'stmt_novelty_cosine',
              'fomc_dissent_count', 'fomc_gov_dissent',
              'fomc_pres_dissent', 'fomc_any_dissent']

print(f"Meeting date rows: {is_meeting.sum()}")
for c in check_cols:
    n = df.loc[is_meeting, c].notna().sum()
    print(f"  {c:<30}: {n}/{is_meeting.sum()}")